# Dataset stats

This is a notebook to get statistics (size, composition) on each dataset we have, so we can better contexualize our findings and potentially spot shortcomings.

In [1]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl
from pathlib import Path
from scipy.stats import mode
from project.utils.functions import load_config

### Specify what datasets to look at

In [3]:
datasets = [
    # "protein properties"
	"prot_param",
    "uniprot_topology",
    # Dependent largely on linear sequence (?)
	"interpro_conserved_site",
	"interpro_repeat",
    "uniprot_peptide",
    "uniprot_post_translational_modification",
    "uniprot_phosphorylation",
    "uniprot_lipidation",
	"biomap_localization_prediction",
    # # Secondary structure
    "biomap_ssp_q3",
    "biomap_ssp_q8",
	"uniprot_secondary_structure",
    # # Dependent largely on tertiary structure
    "uniprot_functional_sites",
	"interpro_binding_site",
	"biomap_metal_ion_binding",
	"interpro_active_site",
    "interpro_domain",
    "interpro_family",
    "interpro_homologous_superfamily",
    # # Dependent on cellular context
    "GO_cc",
    "GO_mf",
    "GO_bp",
]

### Read datasets and extract size, train:val:test counts, and other stats

In [4]:

dataset_stats = []
label_distributions = []

for dataset_name in datasets:
    try:
        config = load_config(dataset_name)
        df = pl.read_parquet(config['data_path'].replace('/network/scratch/s', '/home/mila/s'))
        if 'targets' in df.columns:
            feats = len(df['targets'].list.explode().unique())
        else:   
            feats = len([c for c in df.columns if df[c].dtype != pl.String])

        samples = len(df)
        frac_train= df['split'].value_counts(normalize=True).filter(pl.col('split') == 'train')['proportion'].item()
        frac_val = df['split'].value_counts(normalize=True).filter(pl.col('split') == 'val')['proportion'].item()
        frac_test = df['split'].value_counts(normalize=True).filter(pl.col('split') == 'test')['proportion'].item()
        if config['task_level'] == 'protein_level':
            annotations_per_protein = df[[c for c in df.columns if (df[c].dtype==pl.Boolean)]].sum_horizontal()
            average_num_annotations = annotations_per_protein.mean()
            std_num_annotations = annotations_per_protein.std()
            annotation_group_sizes = df[[c for c in df.columns if (df[c].dtype==pl.Boolean)]].sum().to_numpy()
            smallest_group_size = annotation_group_sizes.min()
            biggest_group_size = annotation_group_sizes.max()
            mode_group_size = mode(annotation_group_sizes, axis=1, keepdims=False).mode.item()
            group_sizes_mean = annotation_group_sizes.mean()
            group_sizes_std = annotation_group_sizes.std()
        else:
            # We'll just fudge this instead of counting annotations at amino acid level, because they're not really comparable (e.g. all amino acids in a domain)
            average_num_annotations, std_num_annotations = 0.0, 0.0
            smallest_group_size, biggest_group_size, mode_group_size, group_sizes_mean, group_sizes_std = 0.0, 0.0, 0.0, 0.0, 0.0

        dataset_stats.append( {
            'dataset': config['dataset_name'],
            'task_level': config['task_level'],
            'task_type': config['task_type'],
            'num_samples': samples,
            'num_features': feats,
            'average_num_annotations': f"{average_num_annotations:.2f} \u00B1 {std_num_annotations:.2f}",
            'group_min_mode_max': f"[{smallest_group_size},{mode_group_size},{biggest_group_size}]",
            'average_group_size': f"{group_sizes_mean:.2f}",
            'split': f"{frac_train:.2f} : {frac_val:.2f} : {frac_test:.2f}",
        })
    except Exception as e:

        dataset_stats.append( {
            'dataset': config['dataset_name'],
            'task_level': config['task_level'],
            'task_type': config['task_type'],
            'num_samples': samples,
            'num_features': feats,
            'average_num_annotations': 0.0,
            'group_min_max': f"[0,0]",
            'average_group_size': f"{0.0}",
            'split': f"{frac_train:.2f} : {frac_val:.2f} : {frac_test:.2f}",
        })
        
        print(dataset_name)
        print(e)
        continue

prot_param
`std` operation not supported for dtype `str`
biomap_localization_prediction
`std` operation not supported for dtype `str`
biomap_metal_ion_binding
`std` operation not supported for dtype `str`


In [5]:
pl.DataFrame(dataset_stats)

dataset,task_level,task_type,num_samples,num_features,average_num_annotations,group_min_max,average_group_size,split,group_min_mode_max
str,str,str,i64,i64,str,str,str,str,str
"""prot_param""","""protein_level""","""regression""",18126,31,"""0""","""[0,0]""","""0.0""","""0.70 : 0.15 : 0.14""",null
"""uniprot_topology""","""amino_acid_level""","""multiclass_classification""",4777,17,"""0.00 ± 0.00""",null,"""0.00""","""0.71 : 0.16 : 0.14""","""[0.0,0.0,0.0]"""
"""interpro_conserved_site""","""protein_level""","""binary_classification""",2948,334,"""1.07 ± 0.29""",null,"""9.49""","""0.70 : 0.15 : 0.15""","""[2,2,185]"""
"""interpro_repeat""","""protein_level""","""binary_classification""",1189,65,"""1.28 ± 0.50""",null,"""23.37""","""0.69 : 0.16 : 0.15""","""[2,2,209]"""
"""uniprot_peptide""","""amino_acid_level""","""multiclass_classification""",4977,4,"""0.00 ± 0.00""",null,"""0.00""","""0.70 : 0.14 : 0.16""","""[0.0,0.0,0.0]"""
…,…,…,…,…,…,…,…,…,…
"""interpro_family""","""protein_level""","""binary_classification""",3601,239,"""1.25 ± 0.53""",null,"""18.89""","""0.58 : 0.19 : 0.23""","""[2,4,716]"""
"""interpro_homologous_superfamil…","""protein_level""","""binary_classification""",8198,100,"""1.44 ± 0.67""",null,"""117.81""","""0.67 : 0.17 : 0.16""","""[13,21,844]"""
"""GO_cc""","""protein_level""","""binary_classification""",16443,1360,"""4.35 ± 3.63""",null,"""52.59""","""0.69 : 0.15 : 0.16""","""[0,2,4990]"""


We've noticed that for some datasets, taking normalized embeddings and running them through a probe that was trained on original embeddings actually seems to give higher performance on our output metric (matthews correlation coefficient). Is there a correlation between particular datasets, and whether the norm was higher or lower?

In [6]:
# Identify which datasets have norm higher than original embeddings
norm_higher_map = {
    'prot_param': False,
    'uniprot_lipidation': True,
    'uniprot_topology': True,
    'uniprot_peptide': True,
    'uniprot_functional_sites': True,
    'uniprot_phosphorylation': False,
    'uniprot_secondary_structure': True,
    'biomap_ssp_q3': True,
    'biomap_ssp_q8': True,
    'biomap_metal_ion_binding': False,
    'biomap_localization_prediction': False,
    'uniprot_post_translational_modification': True,
    'interpro_binding_site': False,
    'interpro_active_site': False,
    'interpro_repeat': False,
    'interpro_conserved_site': False,
    'interpro_family': False,
    'interpro_homologous_superfamily': False,
    'interpro_domain': False,
    'GO_mf': False,
    'GO_bp': False,
    'GO_cc': False
}

# We need to add a categorical column to be able to do correlations
data_summary_df = pl.DataFrame(dataset_stats).with_columns(
    pl.col('task_level').cast(pl.Categorical).cast(pl.Int16).alias("task_level_categorical"),
    pl.col('task_type').cast(pl.Categorical).cast(pl.Int16).alias("task_type_categorical"),
    ).sort(by=['task_level', 'task_type', 'num_samples']).with_columns(
        pl.col('dataset').replace_strict(norm_higher_map).alias('norm_higher')
    ).with_columns(
        ~pl.col('norm_higher').alias('norm_lower')
    )

In [7]:
data_summary_df

dataset,task_level,task_type,num_samples,num_features,average_num_annotations,group_min_max,average_group_size,split,group_min_mode_max,task_level_categorical,task_type_categorical,norm_higher,norm_lower
str,str,str,i64,i64,str,str,str,str,str,i16,i16,bool,bool
"""uniprot_lipidation""","""amino_acid_level""","""multiclass_classification""",978,25,"""0.00 ± 0.00""",null,"""0.00""","""0.67 : 0.13 : 0.20""","""[0.0,0.0,0.0]""",1,3,true,false
"""uniprot_topology""","""amino_acid_level""","""multiclass_classification""",4777,17,"""0.00 ± 0.00""",null,"""0.00""","""0.71 : 0.16 : 0.14""","""[0.0,0.0,0.0]""",1,3,true,false
"""uniprot_peptide""","""amino_acid_level""","""multiclass_classification""",4977,4,"""0.00 ± 0.00""",null,"""0.00""","""0.70 : 0.14 : 0.16""","""[0.0,0.0,0.0]""",1,3,true,false
"""uniprot_functional_sites""","""amino_acid_level""","""multiclass_classification""",6893,6,"""0.00 ± 0.00""",null,"""0.00""","""0.69 : 0.16 : 0.15""","""[0.0,0.0,0.0]""",1,3,true,false
"""uniprot_phosphorylation""","""amino_acid_level""","""multiclass_classification""",8648,6,"""0.00 ± 0.00""",null,"""0.00""","""0.70 : 0.15 : 0.15""","""[0.0,0.0,0.0]""",1,3,false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GO_mf""","""protein_level""","""binary_classification""",11504,2673,"""3.74 ± 2.93""",null,"""16.10""","""0.70 : 0.16 : 0.15""","""[0,2,1843]""",0,4,false,true
"""GO_bp""","""protein_level""","""binary_classification""",12255,8192,"""8.24 ± 10.76""",null,"""12.32""","""0.70 : 0.15 : 0.14""","""[0,2,1305]""",0,4,false,true
"""GO_cc""","""protein_level""","""binary_classification""",16443,1360,"""4.35 ± 3.63""",null,"""52.59""","""0.69 : 0.15 : 0.16""","""[0,2,4990]""",0,4,false,true


In [ ]:
data_summary_df.write_csv(Path('/home/mila/s/shawn.whitfield/scratch/results/dataset_characterization') / 'datasets_summary.csv')

In [9]:
data_summary_df[['task_level_categorical', 'task_type_categorical', 'num_samples', 'num_features', 'norm_higher', 'norm_lower']].corr()

task_level_categorical,task_type_categorical,num_samples,num_features,norm_higher,norm_lower
f64,f64,f64,f64,f64,f64
1.0,-0.649722,0.059361,-0.286899,0.908514,-0.908514
-0.649722,1.0,-0.366981,0.321162,-0.590281,0.590281
0.059361,-0.366981,1.0,0.281112,0.044992,-0.044992
-0.286899,0.321162,0.281112,1.0,-0.260466,0.260466
0.908514,-0.590281,0.044992,-0.260466,1.0,-1.0
-0.908514,0.590281,-0.044992,0.260466,-1.0,1.0


It looks like there's a strong correlation between the task level and whether the norm is higher or lower.